In [182]:
import torch.nn as nn
import torch.utils.data.dataloader
from torch.nn.functional import scaled_dot_product_attention
import numpy as np
from src.language_models.utils import repackage_hidden, get_batch, batchify, save_checkpoint, move_to_device, save_val_loss_data
from src.language_models.dictionary_corpus import Corpus
from tqdm import tqdm

# Model

In [164]:
class CBR_RNN(nn.Module): 
# goal here is to reuse CBR_RNN but with scaled dot product attention for more efficient computations. 
# Also I got rid of options such as loading pretrained embeddings, and ablating attention to simplify the code.
# In the future if those options are needed, they can still be copy pasted from William's code as the structure hasn't changed
    def __init__(self, ntoken, ninp, nhid, dropout=0.5, device=None):
        super().__init__()
        #same layers as Timkey
        self.device = device
        self.tanh = nn.Tanh()
        self.drop = nn.Dropout(dropout)
        self.score_attn = nn.Softmax(dim=-1)
        self.encoder = nn.Embedding(ntoken, ninp)
        self.q = nn.Linear(ninp+nhid,nhid)
        self.intermediate_h = nn.Linear(nhid*4,nhid*4)
        self.decoder = nn.Linear(nhid, ntoken+1)
        self.q_norm = torch.nn.LayerNorm(nhid)
        self.int_norm = torch.nn.LayerNorm(nhid * 4)
        self.f_norm = torch.nn.LayerNorm(nhid * 3)  
        self.nhid = nhid
        self.attn_div_factor = np.sqrt(nhid)
        self.final_h = nn.Linear(nhid*4,nhid*3)
        
        
    #same weight initialization as Timkey
    def init_weights(self, freeze_embedding, aux_objective):
        """ Initialize encoder and decoder weights """
        initrange = 0.1
        if not freeze_embedding:
            self.encoder.weight.data.uniform_(-initrange, initrange)
        self.decoder.bias.data.fill_(0)
        self.decoder.weight.data.uniform_(-initrange, initrange)
        if(aux_objective):
            self.aux_decoder.bias.data.fill_(0)
            self.aux_decoder.weight.data.uniform_(-initrange, initrange)

    
    def init_hidden(self, bsz):
        """ Initialize a fresh hidden state """
        weight = next(self.parameters()).data
    
        return torch.tensor(weight.new(bsz, self.nhid).zero_())
    
    def init_cache(self, observation):
        if len(observation.size())>1:
            bsz = observation.size(dim=-1)
        else:
            bsz = 1
        seq_len = observation.size(dim=0)

        return torch.zeros(1, bsz, self.nhid).to(self.device), torch.zeros(1, bsz, self.nhid).to(self.device), torch.zeros(1, bsz, self.nhid).to(self.device)


    def forward(self, observation, initial_cache, attention_mask=None):
        # Get dimensions
        seq_len = observation.size(0) #if len(observation.size()) > 1 else 1
        # Unpack initial cache
        hidden, key_cache, value_cache = initial_cache
        
        # 1. Encode observations
        emb = self.drop(self.encoder(observation))
        new_hidden_states = []
        # Process sequence : is there another more efficient way to compute causal attention than looping ?
        for i in range(seq_len): #need to keep sequential processing as the core structure is recurrent (each new word needs the hidden state obtained after prediction of the last word)
            # 2. Concatenate with previous hidden state
            
            query = self.drop(self.tanh(self.q_norm(self.q(torch.cat((emb[i],hidden[i]), -1))))) #b * d
            query = query.unsqueeze(1) 
            # query_n = query.unsqueeze(-1) #b * n * 1
            # print(query_n.shape)
            # Apply attention : for the mask we directly use the causal attention mask from pytorch here
            attn_output = scaled_dot_product_attention(
                query, key_cache.transpose(0, 1), value_cache.transpose(0, 1),
                is_causal=True
            )
            attn = attn_output.squeeze(1)
          
            intermediate = self.drop(self.tanh(self.int_norm(self.intermediate_h(torch.cat((emb[i],query.squeeze(1),attn,hidden[i]),-1)))))
            key_cache_i, value_cache_i, hidden_i = self.drop(self.tanh(self.f_norm(self.final_h(intermediate)))).split(self.nhid, dim=-1)
            
            hidden = torch.cat((hidden, hidden_i.unsqueeze(0)), dim=0)
            key_cache = torch.cat((key_cache, key_cache_i.unsqueeze(0)), dim=0)

            value_cache = torch.cat((value_cache, value_cache_i.unsqueeze(0)), dim=0)
            new_hidden_states.append(hidden_i)
          
        #output = hidden[1:]
        output_hidden = torch.stack(new_hidden_states)
        decoded = self.decoder(output_hidden)
        
        return decoded, hidden

# Training function

In [178]:
def train(model, criterion, train_data, batch_size, ntokens):
    # Turn on training mode which enables dropout.
    model.train()
    total_loss = 0
    #NEW : move hidden to devide
    hidden = model.init_hidden(batch_size)

    for batch, i in enumerate(tqdm(range(0, train_data.size(0) - 1, 35), desc="Training")):
        data, targets = get_batch(train_data, i, 35)
        #NEW : move data and target to device
        cache = model.init_cache(data)
        # truncated BPP
        hidden = repackage_hidden(hidden)
        model.zero_grad()
        output, hidden = model(data, cache)
        output_flat = output.reshape(-1, output.size(-1))
        
        # Similarly, reshape targets to [seq_len*batch_size]
        targets_flat = targets.reshape(-1)
        #loss = criterion(output.view(-1, ntokens), targets)
        loss=criterion(output_flat, targets_flat)
        loss.backward()

        # `clip_grad_norm` helps prevent the exploding gradient problem in RNNs / LSTMs.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
        for p in model.parameters():
            p.data.add_(-10, p.grad.data)

        total_loss += loss.item()

# Data

In [10]:
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')


In [45]:
ntokens = len(corpus.dictionary)


In [46]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [47]:
train_data = batchify(corpus.train, 128, device)
val_data = batchify(corpus.valid, 128, device)
test_data = batchify(corpus.test, 128, device)

In [48]:
criterion = nn.CrossEntropyLoss()


In [174]:
model = CBR_RNN(ntokens, 128, 128)

In [175]:
batch_size = 128

In [183]:
train(model, criterion, train_data, batch_size, ntokens)

/tmp/ipykernel_800318/264192033.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(weight.new(bsz, self.nhid).zero_())
Training:   0%|          | 25/18540 [00:14<2:54:53,  1.76it/s]


KeyboardInterrupt: 